# Lyα-forest tomography with a literature-style Wiener filter

This notebook reconstructs a three-dimensional Lyα transmitted-flux field from
sparse, noisy sightlines through a held-out CAMELS volume.

The central observable is the **flux contrast**

$$
\delta_F(\mathbf{x}) = \frac{F(\mathbf{x})}{\langle F\rangle}-1.
$$

This is the field reconstructed in the standard Lyα-tomography literature.  A
Wiener filter is an optimal *linear* estimator when the adopted signal covariance
and noise model are adequate.  It does not directly invert flux into dark-matter
density.  We therefore assess the flux reconstruction first and only then show how
the recovered absorption traces the smoothed density field.

## What has been improved

The reconstruction follows five practices used in successful Lyα tomography:

1. **Reconstruct $\delta_F$, not raw flux or nonlinear density.**
2. **Propagate the spectral noise into $\delta_F$ units.**
3. **Choose correlation lengths near the mean sightline separation.**
4. **Tune hyperparameters only on separate validation simulations.**
5. **Compare truth and reconstruction at the attainable map resolution**, rather
   than expecting the sparse data to recover the native simulation grid.

The final statistics use seven held-out simulation realizations.  The parameters
are never selected using those test boxes.

## 1. Imports and configuration

The simulation volume is periodic.  We therefore use periodic distances and
periodic smoothing.  A real survey is not periodic; production pipelines instead
reconstruct a buffered region and discard its edges.

In [ ]:
from pathlib import Path
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from scipy import ndimage, special
from astropy import units as u
from astropy.constants import c, k_B, m_e, m_p
from astropy.cosmology import Planck18 as cosmo

rng = np.random.default_rng(2026)

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'semibold',
    'axes.labelsize': 10.5,
    'legend.frameon': False,
})

notebook_directory = Path('.') if Path('Sims').exists() else Path('Hands-On')
data_directory = notebook_directory / 'Sims/CMD_z=2_grid128'
output_directory = notebook_directory / 'figures'
output_directory.mkdir(exist_ok=True)

box_size = 25.0                 # h^-1 cMpc
redshift = 2.0
calibration_simulations = np.arange(18)
validation_simulations = np.arange(18, 20)
test_simulations = np.arange(20, 27)

target_transverse_spacing = 2.0  # h^-1 cMpc
signal_to_noise = 5.0             # per native flux pixel
instrument_fwhm_kms = 120.0

rebin_factor = 4
literature_smoothing_factor = 1.4

### Conceptual pipeline

The neural and FGPA notebooks address **one-dimensional physical inversion**.
This notebook addresses the complementary **three-dimensional interpolation**
problem.

In [ ]:
labels = [
    ('Sparse noisy\nLyα spectra', '#DCEAF7'),
    (r'$\delta_F=F/\langle F\rangle-1$', '#E7F2E4'),
    ('Wiener posterior\nmean', '#F8E8C8'),
    ('3D flux-contrast\nmap', '#E7DDF2'),
]

fig, axis = plt.subplots(figsize=(12, 2.5), constrained_layout=True)
axis.set_xlim(0, 12)
axis.set_ylim(0, 2.5)
axis.axis('off')

x_positions = [0.25, 3.25, 6.25, 9.25]
for x, (label, colour) in zip(x_positions, labels):
    box = FancyBboxPatch(
        (x, 0.75), 2.4, 1.0,
        boxstyle='round,pad=0.04,rounding_size=0.08',
        facecolor=colour, edgecolor='#333333', linewidth=1.1,
    )
    axis.add_patch(box)
    axis.text(x + 1.2, 1.25, label, ha='center', va='center', fontsize=11)

for x in [2.68, 5.68, 8.68]:
    axis.add_patch(FancyArrowPatch((x, 1.25), (x + 0.5, 1.25),
                                  arrowstyle='-|>', mutation_scale=14,
                                  color='#444444', linewidth=1.2))

axis.text(6.0, 0.25,
          'The map is evaluated and interpreted at the sampling-limited resolution.',
          ha='center', fontsize=10, color='#444444')
plt.show()

## 2. Load the CAMELS fields and define the map grid

The native grid is much finer than the resolution supported by sightlines separated
by roughly $2\ h^{-1}\,\mathrm{cMpc}$.  We therefore reconstruct on a coarser grid
whose cell width remains smaller than the final map resolution.  This greatly
reduces the matrix size without removing measurable information.

In [ ]:
gas_boxes = np.load(data_directory / 'Grids_Mgas_IllustrisTNG_CV_128_z=2.0.npy', mmap_mode='r')
dm_boxes = np.load(data_directory / 'Grids_Mcdm_IllustrisTNG_CV_128_z=2.0.npy', mmap_mode='r')
HI_boxes = np.load(data_directory / 'Grids_HI_IllustrisTNG_CV_128_z=2.0.npy', mmap_mode='r')
temperature_boxes = np.load(data_directory / 'Grids_T_IllustrisTNG_CV_128_z=2.0.npy', mmap_mode='r')

N_native = gas_boxes.shape[1]
assert N_native % rebin_factor == 0

dx_native = box_size / N_native
N_map = N_native // rebin_factor
dx_map = box_size / N_map

native_coordinates = (np.arange(N_native) + 0.5) * dx_native
map_coordinates = (np.arange(N_map) + 0.5) * dx_map
map_native_indices = np.arange(rebin_factor // 2, N_native, rebin_factor)

gas_means = np.array([box.mean(dtype=np.float64) for box in gas_boxes])
dm_means = np.array([box.mean(dtype=np.float64) for box in dm_boxes])

number_of_sightlines = round((box_size / target_transverse_spacing) ** 2)
flat_indices = rng.choice(N_native * N_native, size=number_of_sightlines, replace=False)
sightline_indices = np.column_stack(np.divmod(flat_indices, N_native))
sightline_xy = (sightline_indices + 0.5) * dx_native

mean_transverse_spacing = np.sqrt(box_size**2 / number_of_sightlines)
map_smoothing_scale = literature_smoothing_factor * mean_transverse_spacing

print(f'Native grid: {N_native}^3; Wiener map grid: {N_map}^3')
print(f'{number_of_sightlines} sightlines; sqrt(A/N_los) = '
      f'{mean_transverse_spacing:.2f} h^-1 cMpc')
print(f'Analysis smoothing scale = {map_smoothing_scale:.2f} h^-1 cMpc')

### Sampling diagnostic

$\sqrt{A/N_{\rm LOS}}$ is the conventional global spacing.  The nearest-neighbour
distribution additionally reveals holes and close pairs in a random sightline
pattern.

In [ ]:
def periodic_distance_2d(points_a, points_b):
    separation = np.abs(points_a[:, None, :] - points_b[None, :, :])
    separation = np.minimum(separation, box_size - separation)
    return np.sqrt(np.sum(separation**2, axis=-1))

line_separations = periodic_distance_2d(sightline_xy, sightline_xy)
np.fill_diagonal(line_separations, np.inf)
nearest_separations = line_separations.min(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)
axes[0].scatter(sightline_xy[:, 0], sightline_xy[:, 1], s=20,
                color='#276FBF', alpha=0.8)
axes[0].set(xlim=(0, box_size), ylim=(0, box_size), aspect='equal',
            xlabel=r'$x\ [h^{-1}\,\mathrm{cMpc}]$',
            ylabel=r'$y\ [h^{-1}\,\mathrm{cMpc}]$',
            title='Random transverse sampling')

axes[1].hist(nearest_separations, bins=14, color='#7A5195', alpha=0.85)
axes[1].axvline(nearest_separations.mean(), color='black', ls='--',
                label=f'mean = {nearest_separations.mean():.2f}')
axes[1].set(xlabel=r'Nearest sightline distance $[h^{-1}\,\mathrm{cMpc}]$',
            ylabel='Number of sightlines', title='Local sampling distribution')
axes[1].legend()
plt.show()

## 3. Make realistic mock spectra

Each observed sightline is generated from the co-spatial CAMELS neutral-hydrogen
and temperature fields.  Every absorber cell contributes a thermally broadened
Voigt profile; the resulting flux is convolved with the instrumental line-spread
function and Gaussian pixel noise is added.

Peculiar velocities are not included because the required velocity grid is not in
this teaching data set.  Thus the mock includes thermal and instrumental broadening
but not redshift-space distortions.  This limitation should be stated explicitly
when interpreting the line-of-sight covariance.

In [ ]:
hydrogen_mass_g = (m_p + m_e).to_value(u.g)
density_unit = (u.Msun / cosmo.h) / (u.Mpc / cosmo.h) ** 3
H_z = cosmo.H(redshift).to_value(u.km / u.s / u.Mpc)

velocity_per_hmpc = H_z / (cosmo.h * (1 + redshift))
velocity_pixel_width = velocity_per_hmpc * dx_native
instrument_sigma_pixels = (
    instrument_fwhm_kms / (2 * np.sqrt(2 * np.log(2))) / velocity_pixel_width
)
flux_noise_std = 1 / signal_to_noise

def number_density_HI(rho_HI):
    comoving_density = (rho_HI * density_unit).to_value(u.g / u.cm**3)
    return comoving_density * (1 + redshift)**3 / hydrogen_mass_g

def voigt_tau_batch(n_HI, temperature):
    c_cms = c.to_value(u.cm / u.s)
    oscillator_cross_section = 4.45e-18
    gamma_alpha = 6.262e8
    nu_alpha = c_cms / (1215.67e-8)

    doppler_b = np.sqrt(2 * k_B.to_value(u.erg / u.K)
                        * temperature / hydrogen_mass_g)
    damping = gamma_alpha * c_cms / (4 * np.pi * nu_alpha * doppler_b)

    source_minus_pixel = native_coordinates[:, None] - native_coordinates[None, :]
    source_minus_pixel = (source_minus_pixel + box_size / 2) % box_size - box_size / 2
    velocity = source_minus_pixel * velocity_per_hmpc * 1e5

    argument = velocity[None, :, :] / doppler_b[:, :, None]
    argument = argument + 1j * damping[:, :, None]
    profile = np.real(special.wofz(argument))

    cell_width_cm = dx_native * u.Mpc.to(u.cm) / cosmo.h
    prefactor = (c_cms * oscillator_cross_section * cell_width_cm
                 * n_HI / (np.sqrt(np.pi) * doppler_b * (1 + redshift)))
    return np.sum(prefactor[:, :, None] * profile, axis=1)

def make_flux(rho_HI, temperature, add_noise, batch_size=32):
    flux = np.empty(rho_HI.shape, dtype=np.float64)
    for start in range(0, len(rho_HI), batch_size):
        stop = min(start + batch_size, len(rho_HI))
        tau = voigt_tau_batch(
            number_density_HI(rho_HI[start:stop]),
            temperature[start:stop],
        )
        intrinsic_flux = np.exp(-tau)
        flux[start:stop] = ndimage.gaussian_filter1d(
            intrinsic_flux, instrument_sigma_pixels, axis=1, mode='wrap'
        )

    if add_noise:
        flux += rng.normal(0, flux_noise_std, size=flux.shape)
    return flux

def rebin_spectra(spectra):
    return spectra.reshape(len(spectra), N_map, rebin_factor).mean(axis=2)

def extract_fields(simulation, transverse_indices):
    ix, iy = transverse_indices.T
    rho_HI = HI_boxes[simulation, ix, iy, :].astype(np.float64)
    temperature = temperature_boxes[simulation, ix, iy, :].astype(np.float64)
    return rho_HI, temperature

### Estimate the mean flux and signal variance without test leakage

An observational analysis would adopt or fit an external mean-flux model.  Here it
is estimated from independent calibration simulations.  The test volumes do not
determine either $\langle F\rangle$ or the Wiener covariance amplitude.

In [ ]:
calibration_flux = []
calibration_lines_per_box = 24

for simulation in calibration_simulations:
    selected = rng.choice(N_native * N_native,
                          size=calibration_lines_per_box, replace=False)
    calibration_indices = np.column_stack(np.divmod(selected, N_native))
    rho_HI, temperature = extract_fields(simulation, calibration_indices)
    calibration_flux.append(rebin_spectra(make_flux(rho_HI, temperature,
                                                     add_noise=False)))

calibration_flux = np.concatenate(calibration_flux)
mean_flux_model = calibration_flux.mean()
calibration_deltaF = calibration_flux / mean_flux_model - 1
signal_variance = calibration_deltaF.var(dtype=np.float64)

rebinned_noise_std_deltaF = (
    flux_noise_std / np.sqrt(rebin_factor) / mean_flux_model
)
base_noise_variance = rebinned_noise_std_deltaF**2

print(f'Calibration <F> = {mean_flux_model:.3f}')
print(f'Var(delta_F) = {signal_variance:.4f}')
print(f'Rebinned pixel-noise sigma(delta_F) = {rebinned_noise_std_deltaF:.3f}')

### Inspect spectra before tomography

The flux is plotted as a continuous noisy spectrum—not as isolated points—because
each sightline is an observed one-dimensional spectral record.

In [ ]:
rho_HI_example, temperature_example = extract_fields(test_simulations[0],
                                                      sightline_indices[:3])
flux_clean_example = make_flux(rho_HI_example, temperature_example, add_noise=False)
flux_noisy_example = make_flux(rho_HI_example, temperature_example, add_noise=True)

fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True,
                         constrained_layout=True)
for index, axis in enumerate(axes):
    axis.plot(native_coordinates, flux_noisy_example[index],
              color='#4C78A8', lw=0.9, label='Noisy observed spectrum')
    axis.plot(native_coordinates, flux_clean_example[index],
              color='#D1495B', lw=1.5, alpha=0.9, label='Noise-free signal')
    axis.set_ylim(-0.15, 1.25)
    axis.set_ylabel('Flux')
    axis.set_title(f'Sightline {index + 1}')
    axis.grid(alpha=0.15)
axes[0].legend(ncol=2, fontsize=9)
axes[-1].set_xlabel(r'Line-of-sight distance $[h^{-1}\,\mathrm{cMpc}]$')
plt.show()

## 4. Wiener reconstruction

For data vector $\mathbf d$ and map vector $\mathbf m$, the Wiener posterior mean is

$$
\widehat{\mathbf m}
= \mathbf C_{\rm MD}
  \left(\mathbf C_{\rm DD}+\mathbf N\right)^{-1}\mathbf d.
$$

Following common Lyα-tomography work, we use a separable Gaussian covariance,

$$
C(\mathbf r_1,\mathbf r_2)=\sigma_F^2
\exp\!\left[-\frac{\Delta r_\parallel^2}{2L_\parallel^2}\right]
\exp\!\left[-\frac{\Delta r_\perp^2}{2L_\perp^2}\right].
$$

$L_\perp$ should be comparable to the sightline spacing.  $L_\parallel$ also
accounts for spectral smoothing.  The code exploits separability, replacing one
very large dense solve with two small symmetric eigendecompositions.

In [ ]:
def periodic_distance_1d(points_a, points_b):
    separation = np.abs(points_a[:, None] - points_b[None, :])
    return np.minimum(separation, box_size - separation)

def gaussian_covariance_xy(points_a, points_b, transverse_scale):
    distance = periodic_distance_2d(points_a, points_b)
    return np.exp(-0.5 * (distance / transverse_scale)**2)

def gaussian_covariance_z(los_scale):
    distance = periodic_distance_1d(map_coordinates, map_coordinates)
    return signal_variance * np.exp(-0.5 * (distance / los_scale)**2)

def wiener_predict(data_deltaF, data_xy, target_xy,
                   transverse_scale, los_scale, noise_variance):
    covariance_xy = gaussian_covariance_xy(data_xy, data_xy,
                                            transverse_scale)
    covariance_z = gaussian_covariance_z(los_scale)

    eigenvalue_xy, eigenvector_xy = np.linalg.eigh(covariance_xy)
    eigenvalue_z, eigenvector_z = np.linalg.eigh(covariance_z)
    eigenvalue_xy = np.clip(eigenvalue_xy, 0, None)
    eigenvalue_z = np.clip(eigenvalue_z, 0, None)

    transformed_data = eigenvector_xy.T @ data_deltaF @ eigenvector_z
    denominator = (eigenvalue_xy[:, None] * eigenvalue_z[None, :]
                   + noise_variance)
    solved_data = eigenvector_xy @ (transformed_data / denominator) @ eigenvector_z.T

    cross_covariance_xy = gaussian_covariance_xy(
        target_xy, data_xy, transverse_scale
    )
    return cross_covariance_xy @ solved_data @ covariance_z

## 5. Tune only on validation simulations

Literature values provide the physically motivated neighbourhood to search, but
the exact optimum depends on the spectra, sampling, noise, and instrumental
resolution.  We hide 20% of the validation sightlines, reconstruct them from the
remaining 80%, and minimize their error relative to their noise-free spectra.

This is a compact cross-validation exercise—not tuning on the final test maps.

In [ ]:
validation_clean, validation_noisy = [], []
for simulation in validation_simulations:
    rho_HI, temperature = extract_fields(simulation, sightline_indices)
    validation_clean.append(rebin_spectra(make_flux(rho_HI, temperature,
                                                     add_noise=False)))
    validation_noisy.append(rebin_spectra(make_flux(rho_HI, temperature,
                                                     add_noise=True)))

validation_clean = np.asarray(validation_clean) / mean_flux_model - 1
validation_noisy = np.asarray(validation_noisy) / mean_flux_model - 1

shuffled_lines = rng.permutation(number_of_sightlines)
split = int(0.8 * number_of_sightlines)
fit_lines = shuffled_lines[:split]
held_out_lines = shuffled_lines[split:]

instrument_sigma_hmpc = (
    instrument_fwhm_kms / (2 * np.sqrt(2 * np.log(2))) / velocity_per_hmpc
)
expected_los_scale = np.sqrt(max(
    mean_transverse_spacing**2 - instrument_sigma_hmpc**2,
    (0.5 * mean_transverse_spacing)**2,
))

transverse_candidates = mean_transverse_spacing * np.array([0.75, 1.0, 1.25])
los_candidates = expected_los_scale * np.array([0.75, 1.0, 1.25])
noise_multipliers = np.array([0.7, 1.0, 1.4])

validation_results = []
for transverse_scale, los_scale, noise_multiplier in product(
        transverse_candidates, los_candidates, noise_multipliers):
    box_errors = []
    for noisy_flux, clean_flux in zip(validation_noisy, validation_clean):
        prediction = wiener_predict(
            noisy_flux[fit_lines], sightline_xy[fit_lines],
            sightline_xy[held_out_lines], transverse_scale, los_scale,
            noise_multiplier * base_noise_variance,
        )
        box_errors.append(np.sqrt(np.mean(
            (prediction - clean_flux[held_out_lines])**2
        )))
    validation_results.append((np.mean(box_errors), transverse_scale,
                               los_scale, noise_multiplier))

validation_results.sort(key=lambda item: item[0])
validation_rmse, best_L_perp, best_L_parallel, best_noise_multiplier = validation_results[0]
effective_noise_variance = best_noise_multiplier * base_noise_variance

print('Selected without using a test map:')
print(f'  L_perp     = {best_L_perp:.2f} h^-1 cMpc')
print(f'  L_parallel = {best_L_parallel:.2f} h^-1 cMpc')
print(f'  noise multiplier = {best_noise_multiplier:.2f}')
print(f'  held-out validation RMSE(delta_F) = {validation_rmse:.4f}')

### Validation landscape

A broad minimum is healthier than a sharply fine-tuned optimum.  The heatmap below
fixes the noise multiplier at its selected value and shows the dependence on the
two physical correlation lengths.

In [ ]:
validation_grid = np.full((len(los_candidates), len(transverse_candidates)), np.nan)
for error, transverse_scale, los_scale, noise_multiplier in validation_results:
    if np.isclose(noise_multiplier, best_noise_multiplier):
        row = np.flatnonzero(np.isclose(los_candidates, los_scale))[0]
        column = np.flatnonzero(np.isclose(transverse_candidates,
                                           transverse_scale))[0]
        validation_grid[row, column] = error

fig, axis = plt.subplots(figsize=(6.8, 5), constrained_layout=True)
image = axis.imshow(validation_grid, origin='lower', cmap='viridis', aspect='auto')
axis.set_xticks(range(len(transverse_candidates)),
                [f'{value:.2f}' for value in transverse_candidates])
axis.set_yticks(range(len(los_candidates)),
                [f'{value:.2f}' for value in los_candidates])
axis.set(xlabel=r'$L_\perp\ [h^{-1}\,\mathrm{cMpc}]$',
         ylabel=r'$L_\parallel\ [h^{-1}\,\mathrm{cMpc}]$',
         title='Held-out validation RMSE')
for row in range(len(los_candidates)):
    for column in range(len(transverse_candidates)):
        colour = 'white' if validation_grid[row, column] < np.nanmedian(validation_grid) else 'black'
        axis.text(column, row, f'{validation_grid[row, column]:.3f}',
                  ha='center', va='center', color=colour, fontsize=9)
fig.colorbar(image, ax=axis, label=r'RMSE in $\delta_F$')
plt.show()

## 6. Reconstruct seven independent test volumes

The dense reference field is evaluated on the coarser map grid using the same
physical spectral model.  The observed data still consist only of the sparse
noisy sightlines.  This separation between the hidden reference and the available
data is essential.

In [ ]:
map_x, map_y = np.meshgrid(map_coordinates, map_coordinates, indexing='ij')
map_xy = np.column_stack((map_x.ravel(), map_y.ravel()))
dense_transverse_indices = np.stack(
    np.meshgrid(map_native_indices, map_native_indices, indexing='ij'), axis=-1
).reshape(-1, 2)

true_deltaF_test = []
observed_deltaF_test = []
wiener_deltaF_test = []
nearest_deltaF_test = []

nearest_line_for_voxel = np.argmin(
    periodic_distance_2d(map_xy, sightline_xy), axis=1
)

for simulation in test_simulations:
    # Sparse observed spectra
    rho_HI, temperature = extract_fields(simulation, sightline_indices)
    observed_flux = rebin_spectra(make_flux(rho_HI, temperature,
                                             add_noise=True))
    observed_deltaF = observed_flux / mean_flux_model - 1

    # Dense hidden reference spectra, evaluated only for validation of the map
    dense_HI, dense_temperature = extract_fields(
        simulation, dense_transverse_indices
    )
    dense_flux = rebin_spectra(make_flux(dense_HI, dense_temperature,
                                          add_noise=False))
    true_deltaF = (dense_flux / mean_flux_model - 1).reshape(
        N_map, N_map, N_map
    )

    wiener_map = wiener_predict(
        observed_deltaF, sightline_xy, map_xy,
        best_L_perp, best_L_parallel, effective_noise_variance,
    ).reshape(N_map, N_map, N_map)

    nearest_map = observed_deltaF[nearest_line_for_voxel].reshape(
        N_map, N_map, N_map
    )

    true_deltaF_test.append(true_deltaF)
    observed_deltaF_test.append(observed_deltaF)
    wiener_deltaF_test.append(wiener_map)
    nearest_deltaF_test.append(nearest_map)

true_deltaF_test = np.asarray(true_deltaF_test, dtype=np.float32)
observed_deltaF_test = np.asarray(observed_deltaF_test, dtype=np.float32)
wiener_deltaF_test = np.asarray(wiener_deltaF_test, dtype=np.float32)
nearest_deltaF_test = np.asarray(nearest_deltaF_test, dtype=np.float32)

## 7. Compare at the resolution supported by the data

Caucci-style analyses commonly use an effective smoothing scale near
$1.4\langle d_{\rm LOS}\rangle$.  We apply the *same* Gaussian smoothing to the
hidden truth, the Wiener map, and a simple nearest-sightline baseline.  Smoothing
only the reconstruction would make the comparison unfair.

In [ ]:
smoothing_sigma_pixels = map_smoothing_scale / dx_map

def smooth_map_ensemble(maps):
    return ndimage.gaussian_filter(
        maps,
        sigma=(0, smoothing_sigma_pixels, smoothing_sigma_pixels,
               smoothing_sigma_pixels),
        mode='wrap',
    )

true_smoothed = smooth_map_ensemble(true_deltaF_test)
wiener_smoothed = smooth_map_ensemble(wiener_deltaF_test)
nearest_smoothed = smooth_map_ensemble(nearest_deltaF_test)

def map_metrics(truth, reconstruction):
    truth_flat = truth.ravel()
    reconstruction_flat = reconstruction.ravel()
    correlation = np.corrcoef(truth_flat, reconstruction_flat)[0, 1]
    rmse = np.sqrt(np.mean((reconstruction_flat - truth_flat)**2))
    normalized_rmse = rmse / truth_flat.std()
    slope = np.dot(truth_flat, reconstruction_flat) / np.dot(truth_flat,
                                                              truth_flat)
    return correlation, normalized_rmse, slope

wiener_metrics = np.asarray([
    map_metrics(truth, reconstruction)
    for truth, reconstruction in zip(true_smoothed, wiener_smoothed)
])
nearest_metrics = np.asarray([
    map_metrics(truth, reconstruction)
    for truth, reconstruction in zip(true_smoothed, nearest_smoothed)
])

metric_names = ['Pearson r', 'NRMSE', 'Regression slope']
print('Mean ± standard deviation across seven held-out simulations')
for column, name in enumerate(metric_names):
    print(f'{name:18s}  Wiener: {wiener_metrics[:, column].mean():.3f} ± '
          f'{wiener_metrics[:, column].std(ddof=1):.3f}   '
          f'Nearest: {nearest_metrics[:, column].mean():.3f} ± '
          f'{nearest_metrics[:, column].std(ddof=1):.3f}')

### Visual comparison in two orientations

The transverse slice tests interpolation between sightlines.  The line-of-sight
slice also reveals whether the adopted anisotropic covariance produces artificial
streaks.  All truth and reconstruction panels share one colour scale.

In [ ]:
representative = 0
middle = N_map // 2
truth = true_smoothed[representative]
reconstruction = wiener_smoothed[representative]
residual = reconstruction - truth

flux_limit = np.quantile(np.abs(truth), 0.99)
residual_limit = np.quantile(np.abs(residual), 0.99)

panels = [
    (truth[:, :, middle].T, 'True: transverse slice', flux_limit),
    (reconstruction[:, :, middle].T, 'Wiener: transverse slice', flux_limit),
    (residual[:, :, middle].T, 'Residual', residual_limit),
    (truth[middle, :, :].T, 'True: line-of-sight slice', flux_limit),
    (reconstruction[middle, :, :].T, 'Wiener: line-of-sight slice', flux_limit),
    (residual[middle, :, :].T, 'Residual', residual_limit),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8.5), constrained_layout=True)
images = []
for axis, (panel, title, limit) in zip(axes.flat, panels):
    image = axis.imshow(panel, origin='lower', extent=(0, box_size, 0, box_size),
                        cmap='RdBu_r', vmin=-limit, vmax=limit)
    images.append(image)
    axis.set(title=title,
             xlabel=r'$x$ or $y\ [h^{-1}\,\mathrm{cMpc}]$',
             ylabel=r'$y$ or $z\ [h^{-1}\,\mathrm{cMpc}]$')

axes[0, 1].scatter(sightline_xy[:, 0], sightline_xy[:, 1],
                   s=8, facecolors='none', edgecolors='#222222', linewidths=0.45)
fig.colorbar(images[0], ax=axes[:, :2], label=r'$\delta_F$', shrink=0.88)
fig.colorbar(images[2], ax=axes[:, 2], label=r'$\widehat{\delta}_F-\delta_F$',
             shrink=0.88)
fig.suptitle(rf'Analysis scale: $\sigma={map_smoothing_scale:.1f}\ '
             r'h^{-1}\,\mathrm{cMpc}$', fontsize=14)
plt.show()

### Point-by-point fidelity and baseline comparison

A hexagonal density map is more legible than millions of scatter points.  The
regression slope diagnoses Wiener amplitude shrinkage; a correlation near unity
alone is not sufficient.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)

values = np.concatenate((truth.ravel(), reconstruction.ravel()))
plot_limit = np.quantile(np.abs(values), 0.995)
density_plot = axes[0].hexbin(
    truth.ravel(), reconstruction.ravel(), gridsize=60,
    mincnt=1, bins='log', cmap='magma'
)
axes[0].plot([-plot_limit, plot_limit], [-plot_limit, plot_limit],
             color='white', ls='--', lw=1.2, label='One-to-one')
axes[0].set(xlim=(-plot_limit, plot_limit), ylim=(-plot_limit, plot_limit),
            xlabel=r'True smoothed $\delta_F$',
            ylabel=r'Reconstructed smoothed $\delta_F$',
            title=f'Test simulation {test_simulations[0]}')
axes[0].legend()
fig.colorbar(density_plot, ax=axes[0], label='log10 number of voxels')

x = np.arange(3)
width = 0.34
axes[1].bar(x - width/2, wiener_metrics.mean(axis=0), width,
            yerr=wiener_metrics.std(axis=0, ddof=1), capsize=4,
            label='Wiener', color='#3B82A0')
axes[1].bar(x + width/2, nearest_metrics.mean(axis=0), width,
            yerr=nearest_metrics.std(axis=0, ddof=1), capsize=4,
            label='Nearest sightline', color='#C77966')
axes[1].set_xticks(x, metric_names)
axes[1].set(title='Mean ± 1σ across seven test boxes', ylim=(0, 1.35))
axes[1].axhline(1, color='black', lw=0.8, alpha=0.5)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.2)
plt.show()

## 8. Statistical comparison across the seven test boxes

Three complementary diagnostics are shown:

- the one-point PDF tests the recovered amplitude distribution;
- the power spectrum tests the scale-dependent variance;
- the cross-correlation coefficient
  $r(k)=P_{\rm true,rec}/\sqrt{P_{\rm true}P_{\rm rec}}$ tests phase agreement.

Wiener filtering is expected to suppress poorly constrained small-scale power.
Agreement is only expected on scales larger than the map resolution.

In [ ]:
def isotropic_spectra(truth, reconstruction, edges):
    modes = 2 * np.pi * np.fft.fftfreq(N_map, d=dx_map)
    kx, ky, kz = np.meshgrid(modes, modes, modes, indexing='ij')
    k = np.sqrt(kx**2 + ky**2 + kz**2)

    truth_fourier = np.fft.fftn(truth - truth.mean())
    reconstruction_fourier = np.fft.fftn(reconstruction - reconstruction.mean())
    normalization = dx_map**3 / N_map**3

    power_true = normalization * np.abs(truth_fourier)**2
    power_reconstruction = normalization * np.abs(reconstruction_fourier)**2
    power_cross = normalization * np.real(
        truth_fourier * np.conj(reconstruction_fourier)
    )

    binned_true, binned_reconstruction, correlation = [], [], []
    for low, high in zip(edges[:-1], edges[1:]):
        mask = (k >= low) & (k < high)
        p_true = power_true[mask].mean()
        p_reconstruction = power_reconstruction[mask].mean()
        p_cross = power_cross[mask].mean()
        binned_true.append(p_true)
        binned_reconstruction.append(p_reconstruction)
        correlation.append(p_cross / np.sqrt(p_true * p_reconstruction))

    return (np.asarray(binned_true), np.asarray(binned_reconstruction),
            np.asarray(correlation))

k_edges = np.geomspace(2 * np.pi / box_size, 2.0, 8)
k_centres = np.sqrt(k_edges[:-1] * k_edges[1:])
spectra = np.asarray([
    isotropic_spectra(truth, reconstruction, k_edges)
    for truth, reconstruction in zip(true_smoothed, wiener_smoothed)
])

pdf_edges = np.linspace(
    np.quantile(true_smoothed, 0.002),
    np.quantile(true_smoothed, 0.998), 34
)
pdf_centres = 0.5 * (pdf_edges[:-1] + pdf_edges[1:])
true_pdfs = np.asarray([
    np.histogram(volume, bins=pdf_edges, density=True)[0]
    for volume in true_smoothed
])
reconstructed_pdfs = np.asarray([
    np.histogram(volume, bins=pdf_edges, density=True)[0]
    for volume in wiener_smoothed
])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), constrained_layout=True)

def mean_with_band(axis, x, values, label, colour):
    mean = values.mean(axis=0)
    standard_deviation = values.std(axis=0, ddof=1)
    axis.plot(x, mean, color=colour, lw=1.8, label=label)
    axis.fill_between(x, mean-standard_deviation, mean+standard_deviation,
                      color=colour, alpha=0.2)

mean_with_band(axes[0], pdf_centres, true_pdfs, 'True', '#222222')
mean_with_band(axes[0], pdf_centres, reconstructed_pdfs, 'Wiener', '#2B7A9B')
axes[0].set(xlabel=r'$\delta_F$', ylabel='PDF', title='One-point distribution')

mean_with_band(axes[1], k_centres, spectra[:, 0, :], 'True', '#222222')
mean_with_band(axes[1], k_centres, spectra[:, 1, :], 'Wiener', '#2B7A9B')
axes[1].set(xlabel=r'$k\ [h\,\mathrm{cMpc}^{-1}]$', ylabel=r'$P_F(k)$',
            title='3D flux power spectrum', xscale='log', yscale='log')

mean_with_band(axes[2], k_centres, spectra[:, 2, :], 'True × Wiener', '#6A4C93')
axes[2].axhline(1, color='black', ls='--', lw=1)
axes[2].axvline(1 / map_smoothing_scale, color='#B44', ls=':', lw=1.2,
                label=r'$1/\sigma_{\rm map}$')
axes[2].set(xlabel=r'$k\ [h\,\mathrm{cMpc}^{-1}]$', ylabel=r'$r(k)$',
            title='Scale-dependent phase fidelity', xscale='log', ylim=(0, 1.08))

for axis in axes:
    axis.grid(alpha=0.2)
    axis.legend(fontsize=9)
fig.suptitle('Mean and 1σ simulation-to-simulation scatter', fontsize=14)
plt.show()

## 9. Relation to the underlying density field

Standard Wiener tomography reconstructs $\delta_F$.  Because stronger absorption
generally occurs in denser regions, $-\delta_F$ is correlated with matter
overdensity after smoothing.  The relationship is neither exact nor a density
inversion: temperature, ionization, thermal broadening, and peculiar velocities
all matter.

The following comparison therefore standardizes both fields and asks only whether
their large-scale morphology agrees.

In [ ]:
def block_average_cube(cube, factor):
    n = cube.shape[0] // factor
    return cube.reshape(n, factor, n, factor, n, factor).mean(axis=(1, 3, 5))

dm_contrast_test = []
for simulation in test_simulations:
    dm_overdensity = (dm_boxes[simulation].astype(np.float64)
                      / dm_means[simulation])
    dm_contrast_test.append(block_average_cube(dm_overdensity, rebin_factor) - 1)
dm_contrast_test = np.asarray(dm_contrast_test, dtype=np.float32)
dm_smoothed = smooth_map_ensemble(dm_contrast_test)

density_flux_correlations = np.asarray([
    np.corrcoef(density.ravel(), (-flux).ravel())[0, 1]
    for density, flux in zip(dm_smoothed, wiener_smoothed)
])

def standardize(field):
    return (field - field.mean()) / field.std()

density_panel = standardize(dm_smoothed[0][:, :, middle]).T
absorption_panel = standardize(-wiener_smoothed[0][:, :, middle]).T
difference_panel = absorption_panel - density_panel

fig, axes = plt.subplots(1, 3, figsize=(14, 4.3), constrained_layout=True)
for axis, panel, title, cmap, limit in [
    (axes[0], density_panel, 'True smoothed matter contrast', 'RdBu_r', 2.5),
    (axes[1], absorption_panel, r'Reconstructed absorption $(-\delta_F)$', 'RdBu_r', 2.5),
    (axes[2], difference_panel, 'Standardized difference', 'coolwarm', 2.5),
]:
    image = axis.imshow(panel, origin='lower', extent=(0, box_size, 0, box_size),
                        cmap=cmap, vmin=-limit, vmax=limit)
    axis.set(title=title, xlabel=r'$x\ [h^{-1}\,\mathrm{cMpc}]$',
             ylabel=r'$y\ [h^{-1}\,\mathrm{cMpc}]$')
    fig.colorbar(image, ax=axis, shrink=0.85)

fig.suptitle('Morphological comparison only: this is not a density inversion',
             fontsize=13)
plt.show()

print('Pearson r between smoothed matter contrast and reconstructed absorption:')
print(f'{density_flux_correlations.mean():.3f} ± '
      f'{density_flux_correlations.std(ddof=1):.3f} across seven test boxes')

## 10. Interpretation

A successful run should show the following behaviour:

- the Wiener map is visibly smoother than the native spectra;
- it outperforms the nearest-sightline baseline in correlation and NRMSE;
- large-scale structures agree in both transverse and line-of-sight slices;
- $r(k)$ is high at $k\lesssim 1/\sigma_{\rm map}$ and declines on smaller scales;
- the reconstructed power is suppressed where the data do not constrain the field;
- reconstructed absorption correlates with smoothed matter, but is not identical to it.

If these conditions are not met, inspect the mean-flux normalization, noise units,
validation curve, line-of-sight axis ordering, and periodic distances before changing
the plotting range.

## 11. Exercises

### Exercise 1 — Vary the sightline density

Repeat the reconstruction for target spacings of 1.5, 2.0, and
$3.0\ h^{-1}\,\mathrm{cMpc}$.  Recompute both $L_\perp$ and the analysis smoothing
scale.  Plot the mean Pearson correlation against spacing.

### Exercise 2 — Vary the spectral quality

Try S/N = 2, 5, and 10.  Explain why gains eventually saturate when transverse
sampling—not pixel noise—becomes the dominant limitation.

### Exercise 3 — Deliberately use the wrong noise units

Replace the $\delta_F$ noise variance by the raw flux variance.  Compare the
regression slope and reconstructed power.  Why does this produce incorrect Wiener
shrinkage?

### Exercise 4 — Test covariance mismatch

Double only $L_\parallel$.  Look for line-of-sight streaking in the $y$–$z$ slice.
Then double only $L_\perp$ and inspect transverse over-smoothing.

### Exercise 5 — Mimic a survey boundary

Replace periodic distances by ordinary distances.  Reconstruct a padded volume and
discard a buffer of at least $2L_\perp$.  Compare the central and edge errors.

### Exercise 6 — Beyond Wiener filtering

Wiener filtering is linear and uses only a two-point covariance.  Investigate a
positivity- or bounds-constrained method such as ORCA, or compare with a learned
nonlinear tomographic reconstruction.  Keep the same train/validation/test split.

## References and choices adopted here

- **Pichon et al. (2001)** introduced Bayesian and constrained-Gaussian
  reconstruction methods for the Lyα forest:
  [arXiv:astro-ph/0105196](https://arxiv.org/abs/astro-ph/0105196).
- **Lee et al. (2014)** formulated observational requirements, reconstructed
  $\delta_F$, set covariance scales from the spectral resolution and sightline
  spacing, and compared maps at a controlled smoothing scale:
  [arXiv:1309.1477](https://arxiv.org/abs/1309.1477).
- **Stark et al. (2015)** described the Wiener estimator and its efficient use for
  Lyα tomographic maps:
  [arXiv:1412.1507](https://arxiv.org/abs/1412.1507).
- **Özbek et al. (2016)** emphasized correlation lengths of order the sightline
  spacing and fair, resolution-matched comparisons:
  [MNRAS 456, 3610](https://academic.oup.com/mnras/article/456/4/3610/1032151).
- **Lee et al. (2018), CLAMATO DR1** used a diagonal pixel-noise covariance,
  $L_\perp=2.5$ and $L_\parallel=2.0\ h^{-1}\,\mathrm{Mpc}$ for a survey with
  approximately $2\ h^{-1}\,\mathrm{Mpc}$ sightline separation:
  [arXiv:1710.02894](https://arxiv.org/abs/1710.02894).
- **Li, Horowitz & Cai (2021)** showed where a constrained nonlinear method can
  improve on the Wiener baseline:
  [arXiv:2102.12306](https://arxiv.org/abs/2102.12306).

## Take-away

**Good Wiener tomography is not obtained by making the covariance arbitrarily
narrow or by comparing to the native simulation grid.**  It comes from filtering
the correct zero-mean field, using a consistent noise covariance, choosing scales
tied to the survey geometry and instrument, and evaluating only at the resolution
the sightlines can support.